## Dataopslag KIA Lengte-effect

Dit is een script om data te uploaden en downloaden van de s3 dataopslag van Deltares, vanaf nu genoemd de 'bucket', voor het KIA Lengte Effecten project.
Voor vragen mail djimin.teng@deltares.nl of bel 06 21838794

### Voorbereiding:
- Installeer een python interpreter (via bijvoorbeeld conda of miniforge) als je dat nog niet hebt

- Installeer de environment met behulp van de environment.yml

### Gebruik:

1. Eerst moet de gebruiker bij het kopje "Toegang" de juiste wachtwoorden invullen om toegang te krijgen tot de bucket.

2. Daarna worden de benodigde functies gedefinieerd, hier hoeft niks aan veranderd te worden (voor extra functies zie contactgegevens bovenaan).

Deze eerste twee stappen moeten in de aangegeven volgorde uitgevoerd worden! 


3. Vervolgens kan de gebruiker onder de verschillende kopjes "Bekijken inhoud bucket", "Data uploaden", "Data downloaden" verschillende acties uitvoeren.


### Toegang
Vul hier de verkregen Access en secret key in tussen de aanhalingstekens (bijvoorbeeld "GeheimWachtwoord")


In [1]:
ACCESS_KEY = " "
SECRET_KEY = " "

### Functies definieren (geen aanpassing nodig!)

In [2]:
# functies definieren

import boto3, os
from botocore.client import Config
from collections import defaultdict


EP_URL = "https://s3.deltares.nl"
BUCKET_NAME = "kia-lengte-effect"

def upload_file_to_s3(file_path, object_name=None):
    """Upload a file to S3."""
    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )

    if object_name is None:
        object_name = os.path.basename(file_path)

    try:
        s3_client.upload_file(file_path, BUCKET_NAME, object_name)
        print(
            f"File '{object_name}' uploaded successfully to '{BUCKET_NAME}/{file_path}'"
        )
    except Exception as e:
        print(f"Error uploading file '{object_name}' to S3: {e}")



def upload_folder(folder_path):
    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )
    main_folder_name = os.path.basename(folder_path)  # Extracts the main folder name

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            relative_path = os.path.relpath(file_path, folder_path).replace("\\", "/")
            s3_key = f"{main_folder_name}/{relative_path}"  # Prepend the main folder name
            s3_client.upload_file(file_path, BUCKET_NAME, s3_key)


def download_folder(s3_folder, local_folder):
    # Replace backslashes with forward slashes in the S3 folder path
    s3_folder = s3_folder.replace("\\", "/")

    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )

    # Ensure the local folder exists
    os.makedirs(local_folder, exist_ok=True)

    # List objects in the specified folder
    objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=s3_folder)

    for obj in objects.get('Contents', []):
        s3_key = obj['Key']
        relative_path = os.path.relpath(s3_key, s3_folder).replace("/", os.sep)
        local_file_path = os.path.join(local_folder, relative_path)
        
        # Ensure the local directory exists
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        
        # Download the file
        s3_client.download_file(BUCKET_NAME, s3_key, local_file_path)

def download_file(s3_path, local_folder):
    # Replace backslashes with forward slashes in the S3 folder path
    s3_path = s3_path.replace("\\", "/")

    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )

    local_path = os.path.join(local_folder,os.path.basename(s3_path)).replace("/", os.sep)

    s3_client.download_file(BUCKET_NAME, s3_path, local_path)

def build_tree(paths):
    tree = lambda: defaultdict(tree)
    root = tree()
    for path in paths:
        parts = path.split('/')
        current_level = root
        for part in parts:
            current_level = current_level[part]
    return root

def print_tree(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + key)
        if isinstance(value, defaultdict):
            print_tree(value, indent + 1)

def show_s3_folders_files(prefix=''):
    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )
    # List objects in the bucket
    objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    paths = [obj['Key'] for obj in objects.get('Contents', [])]

    # Build and print the tree
    tree = build_tree(paths)
    print_tree(tree)



def build_tree2(paths):
    tree = lambda: defaultdict(tree)
    root = tree()
    for path in paths:
        parts = path.split('/')
        current_level = root
        for part in parts[:-1]:  # Exclude the file itself
            current_level = current_level[part]
    return root

def print_tree2(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + key)
        if isinstance(value, defaultdict):
            print_tree2(value, indent + 1)

def show_s3_folders(prefix=''):
    s3_client = boto3.client(
        "s3",
        endpoint_url=EP_URL,
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="eu-west-1",
    )
    # List objects in the bucket
    objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    paths = [obj['Key'] for obj in objects.get('Contents', [])]

    # Build and print the tree
    tree = build_tree2(paths)
    print_tree2(tree)

### Bekijken inhoud bucket

Om te bekijken wat er al geupload is naar de dataopslag kan je de onderstaande code gebruiken.

`show_s3_folders()` toont alle folders in de bucket

Als je alleen geinteresseerd bent in een bepaalde folder, dan kan je door een prefix toe te voegen alleen deze folder binnen de bucket bekijken:

`show_s3_folders(prefix="Test_folder/Test1")` toont dus alle folders in de folder 'Test1'

Het is ook mogelijk om _prefix=_ weg te laten, dus `show_s3_folders("Test_folder/Test1")` is hetzelfde als `show_s3_folders(prefix="Test_folder/Test1")`

`show_s3_folders_files()` toont alle folders EN bestanden in de bucket

**Hierbij geldt dezelfde regel met de prefix toevoegen, wat erg aan te raden is zodra er veel data in de bucket zit!**



In [ ]:
show_s3_folders()

show_s3_folders_files(prefix='Test_folder')

### Data uploaden

Het is mogelijk om:
1. Een bestand te uploaden met `upload_file_to_s3(file_path)`

Hierbij is `file_path` het lokale pad naar het bestand dat je wilt uploaden, bijvoorbeeld:

`upload_file_to_s3(file_path=r"c:\Users\TestNaam\Test.txt")`

dus je moet in `r" "` het pad naar je bestand invullen (de r voor de aanhalingstekens is geen spelfout)


2. Een folder te uploaden met `upload_folder(folder_path)`

Hierbij is `folder_path` het lokale pad naar de folder die je wilt uploaden, bijvoorbeeld:

`upload_folder(folder_path=r"c:\Users\TestNaam\TestFolder")`

Hierbij is het belangrijk dat je geen \ achteraan je pad hebt staan:

- Goed: c:\Users\TestNaam\TestFolder

- Fout: c:\Users\TestNaam\TestFolder\

__Het kan zijn dat het uploaden van grote folders lang duurt, omdat alle inhoud van de folder wordt doorlopen en vervolgens geupload__


In [ ]:
upload_folder(folder_path=r" ")

### Data downloaden
Het is mogelijk om:
1. Een bestand te downloaden met `download_file(s3_path, local_folder)`

`s3_path` pad van het bestand op de bucket, hier staat het bestand in de bucket dat je wilt downloaden

`local_folder` hier komt het bestand terecht op je computer

Een voorbeeld is: 
`download_file(r"Test_folder\Test1\test.txt", r"c:\Users\Testnaam\Testfolder")`

2. Een folder te downloaden met `download_folder(s3_folder, local_folder)`

`s3_folder` pad van de folder op de bucket

`local_folder` hier komt de inhoud van de folder terecht op je computer

Een voorbeeld is: 
`download_folder(r"Test_folder\Test1", r"c:\Users\Testnaam\Testfolder")`


__Ook hier is het belangrijk om de folderpaden niet te laten eindigen op \ (zie ook "Data uploaden")__

De paden van de bestanden of folders die je wilt downloaden kan je op basis van de functies onder "Bekijken inhoud bucket" bepalen.

In [ ]:
download_file(s3_path=r" ", local_folder=r" ")

download_folder(s3_folder=r" ", local_folder=r" ")
